In [0]:
# ── Connection ──

jdbc_url = (
    "jdbc:postgresql://aws-1-ap-southeast-2.pooler.supabase.com:5432/postgres"
    "?sslmode=require"
)

connection_props = {
    "user": "postgres.zcwxwopxpbxpdrzrbuao",  # pooler username format
    "password": "YOUR_SUPABASE_PASSWORD",
    "driver": "org.postgresql.Driver"
}



In [0]:
# ── Test connection ──
spark.read.jdbc(
    url=jdbc_url,
    table="(SELECT 1 as test) as src",
    properties=connection_props
).show()

print("Connection successful")

In [0]:
# ── Watermark helper ──

from pyspark.sql.functions import max as spark_max

def get_last_watermark(table_name, watermark_col):
    try:
        df = spark.table(table_name)
        return df.select(spark_max(watermark_col)).collect()[0][0]
    except:
        return None

def incremental_read(jdbc_url, table, watermark_col, last_value, props):
    if last_value is None:
        query = f"(SELECT * FROM {table}) as src"
    else:
        query = f"""
            (SELECT *
             FROM {table}
             WHERE {watermark_col} > '{last_value}') as src
        """
    return spark.read.jdbc(
        url=jdbc_url,
        table=query,
        properties=props
    )


In [0]:
# ── Ingest bank_customers ──

watermark_col = "created_at"

last_value = get_last_watermark(
    "workspace.bronze.bronze_churn_raw",
    watermark_col
)

print(f"Last watermark: {last_value}")

customers_df = incremental_read(
    jdbc_url,
    "public.bank_customers",
    watermark_col,
    last_value,
    connection_props
)

row_count = customers_df.count()
print(f"New rows to ingest: {row_count}")

if row_count > 0:
    customers_df.write.format("delta") \
        .mode("append") \
        .saveAsTable("workspace.bronze.bronze_churn_raw")
    print(f"Ingested {row_count} new rows into workspace.bronze.bronze_churn_raw")
else:
    print("No new rows to ingest — table is up to date")